# Inference with the rate equations (ODE) vs the NN surrogate

Module `inference/ode_likelihood.py` (15/10/2026).  The likelihood is the one of the workflow,
log L = Σ_d [−MSE_d/(2σ_d²) − log(σ_d√2π)], with the same data, σ_eff and prior as the `main.ipynb` run `RUN_IN`.
Only the tr-PL model curve changes: NN surrogate → legacy ODE solver (`solve_transient`, unchanged; post-processing
as for the training data).  The ss-PL term has no surrogate (`sspl_tf` is the steady-state physics) and is the same
in both; in mode `sspl` the "NN" runs are therefore already the exact model and the ODE part is skipped.

1. **Setup** and **ODE check**: worker start-up, solve throughput (timeit), log L and curves NN vs ODE at the best
   fit and the truth.
2. **NN surrogate** with the sampler settings used for the ODE: tempered SMC and emcee (timeit).
3. **ODE: tempered SMC** (timeit) → samples, stages, corner HDF5 + preview.
4. **ODE: emcee** (timeit) → samples, corner HDF5 + preview.
5. **Comparison**: marginals vs NN NUTS of `RUN_IN`, classifier two-sample tests, overlay corner, HDF5 corners of
   every method (Origin script / `corner_plot`).
6. **Timing table**: wall time, evaluations, time per evaluation, speed-up NN vs ODE for identical settings, and
   the ODE time estimated for the evaluation counts of the full `main.ipynb` samplers.

NUTS is not run with the ODE: it needs gradients of log L, which the legacy solver does not provide.

In [ ]:
# ── Which run ────────────────────────────────────────────────────────────────
CONFIG_PATH = None          # unused: the config snapshot of RUN_IN (settings.json) is used (scripts/run_notebook.py needs it)
RUN_IN = "results/1trap_synthetic/1trap_synthetic_trpl"   # finished main.ipynb run (data, sigma, best fit, NUTS)
MODE = None                 # None = mode of RUN_IN ('trpl' | 'sspl' | 'joint')

# ── Samplers (identical for NN and ODE) ─────────────────────────────────────
SMC = dict(n_particles=4096, n_islands=4, ess_target=0.5, target_accept=0.25, p_move=0.95, min_mh=2, max_mh=30)
EMCEE = dict(n_walkers=40, n_steps=10000, burn_frac=0.5, moves="stretch", seed=1)
RUN_NN = True               # NN runs (SMC; emcee if RUN_NN_EMCEE)
RUN_ODE_SMC = True
RUN_ODE_EMCEE = True
RUN_NN_EMCEE = True         # emcee with the NN (only if RUN_NN)
SIGMA_TRPL = None           # None = sigma of RUN_IN; a number = this tr-PL sigma for NN and ODE alike (e.g. sigma_model
                            # without the network term sigma_nn: the ODE has no surrogate error)

N_JOBS = 8                  # ODE worker processes (rekenbeest: 8 logical CPUs)
MAX_BATCH = 16384           # rows per NN call (fixed shape: one XLA program); >= the SMC population
SEED = 20261015
OUT_TAG = "ode_vs_nn"       # output folder inside RUN_IN
CUDA_DEVICE = "0"
PROGRESS_LOG = None         # text file for ODE progress lines (None = RUN_IN/OUT_TAG/ode_progress.log)

## 0. Imports, network, prior, data, likelihoods

In [ ]:
import os, json, math, time, warnings
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = str(CUDA_DEVICE)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
%matplotlib inline
warnings.filterwarnings("ignore", category=UserWarning)

from inference import surrogate, likelihood, sspl_tf, data, samplers, report, smc, runs, corner_plot as cp
from inference import ode_likelihood as ol
from inference.prior import Prior
from inference.benchmark import timeit_call, format_mean_sd
surrogate.configure_tensorflow()

RUN_IN = Path(RUN_IN)
settings_in = json.loads((RUN_IN / "settings.json").read_text())
cfg = settings_in["config"]
MODE = MODE or settings_in["mode"]
SIG = runs.run_sigmas(RUN_IN)                    # sigma_eff of RUN_IN (sampling_info.json or 6_best_fit.json)
if SIGMA_TRPL is not None:
    SIG = dict(SIG, trpl=float(SIGMA_TRPL))
best = json.loads((RUN_IN / "6_best_fit.json").read_text())
Eg, npulse, T = cfg["physics"]["Eg_eV"], cfg["physics"]["npulse_cm3"], cfg["physics"]["T_K"]
d_cm = cfg["physics"]["thickness_nm"] * 1e-7

art = surrogate.load_network(cfg["nn"]["dir"], cfg["nn"].get("stem"))
prior = Prior.from_artifacts(art, Eg=Eg, order_traps=cfg["prior"]["order_traps"],
                             photodoping_traps=cfg["prior"]["photodoping_traps"],
                             bounds_override=cfg["prior"]["bounds_override"], fixed=cfg["prior"].get("fixed", {}), T=T)
pb_in = pd.read_csv(RUN_IN / "prior_bounds.csv").set_index("parameter")
assert np.allclose(pb_in.loc[prior.names, "theta_lo"], prior.lo) and np.allclose(pb_in.loc[prior.names, "theta_hi"], prior.hi), \
    "prior differs from the prior of RUN_IN"
D, free, keys = prior.dim, list(prior.free_idx), prior.display_keys
HAS_TRPL, HAS_SSPL = MODE in ("trpl", "joint"), MODE in ("sspl", "joint")
bs = cfg["hardware"]["batch_size"]

# tr-PL: the points and sigma_eff of RUN_IN
trpl = pd.read_csv(RUN_IN / "trpl_used.csv")
sur = surrogate.make_trpl_surrogate(art, npulse, Eg, trpl.qfls.values, interp_space=cfg["trpl"]["interp_space"])
term_trpl = likelihood.make_trpl_term(sur, trpl.qfls.values, trpl.tau_diff.values, sigma=SIG["trpl"])
# ss-PL: TF steady-state model (no surrogate), the points and sigma_eff of RUN_IN
term_sspl, LP_SS = None, None
if HAS_SSPL:
    ss_cfg = cfg["sspl"]
    SSPL_MODE = sspl_tf.canonical_mode(ss_cfg["mode"])
    ss = pd.read_csv(RUN_IN / f"sspl_used_{SSPL_MODE}.csv")
    G_PER_SUN = data.sspl_g_per_sun(cfg, art.constants)[0] if SSPL_MODE.endswith("_suns") else None   # Voc,rad or absorptance
    lo, hi, n = ss_cfg["n_sweep"]
    sspl_model = sspl_tf.SsplModel(Eg, prior.n_traps, art.constants["Nc"], art.constants["Nv"], T=T, n_sweep=(lo, hi, n))
    term_sspl = likelihood.SsplTerm(sspl_tf.SsplObservation(sspl_model, SSPL_MODE, ss.x.values, ss.y.values,
                                                            G_per_sun=G_PER_SUN), sigma=SIG["sspl"])
    LP_SS = likelihood.LogPosterior(prior, [], term_sspl, "sspl", batch_size=bs)

(RUN_IN / OUT_TAG).mkdir(parents=True, exist_ok=True)
LP_NN = likelihood.LogPosterior(prior, [term_trpl] if HAS_TRPL else [], term_sspl, MODE, batch_size=bs)
OLP = ol.OdeLogPosterior(prior, MODE, art.constants, npulse, T, sur.qfls_used, term_trpl.y_obs, SIG["trpl"],
                         sspl_log_post=LP_SS, n_jobs=N_JOBS,
                         progress_path=PROGRESS_LOG or RUN_IN / OUT_TAG / "ode_progress.log") if HAS_TRPL else None
LABEL_NN = "NN" if HAS_TRPL else "exact (sspl_tf)"
RUN_ODE = HAS_TRPL and (RUN_ODE_SMC or RUN_ODE_EMCEE)

theta_best = np.array(best["theta_best"], float)
truth_theta = None
if cfg.get("true_params"):
    tp = cfg["true_params"]
    truth_theta = np.atleast_2d(prior.from_physical(tp["Et_eV"], tp["Nt_cm3"], tp["tau_n_s"], tp["tau_p_s"], tp["krad_cm3s"]))[0]
start_theta = np.array(json.loads((RUN_IN / "start_point.json").read_text())["theta"], float) \
    if (RUN_IN / "start_point.json").exists() else None
if start_theta is not None and not prior.in_support(start_theta[None])[0]:
    start_theta = None

nuts_theta, NUTS_REF = None, settings_in["mode"] == MODE and (RUN_IN / "posterior_samples_nuts.csv").exists()
if NUTS_REF:
    nuts_df = pd.read_csv(RUN_IN / "posterior_samples_nuts.csv")
    nuts_theta = nuts_df[prior.names].values
    C = int(nuts_df.chain.max()) + 1
    nuts_res = samplers.SamplerResult("nuts", theta=nuts_theta.reshape(C, -1, D), u=prior.inverse(nuts_theta).reshape(C, -1, D))

OUT = RUN_IN / OUT_TAG
FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)
TIMINGS = []
def log_timing(t, n_evals=None, note="", method=None, model=None):
    row = t.as_dict(); row.update(n_evals=n_evals, note=note, method=method, model=model); TIMINGS.append(row)
    print(t, f"| evals: {n_evals:,}" if n_evals else "")
report.save_json({"run_in": str(RUN_IN), "mode": MODE, "network": cfg["nn"]["dir"], "sigmas": SIG, "smc": SMC, "emcee": EMCEE,
                  "n_jobs": N_JOBS, "cpu_count": os.cpu_count(), "seed": SEED, "nuts_reference": bool(NUTS_REF)},
                 OUT / "settings.json")
print(f"mode {MODE} | σ_trPL = {SIG['trpl']:.4f}, σ_ssPL = {SIG.get('sspl', float('nan')):.4f} | tr-PL points {sur.n_points} | "
      f"NUTS reference from RUN_IN: {NUTS_REF} | ODE workers {N_JOBS} on {os.cpu_count()} logical CPUs | output {OUT}")

## 1. ODE check: worker start-up, throughput, NN vs ODE at the best fit and the truth

The worker processes are started and numba is compiled once (untimed for the samplers).  Throughput: one batch of
256 prior draws (what SMC evaluates first) and 256 draws near the posterior, timeit.

In [ ]:
if HAS_TRPL:
    _, t_w = timeit_call(OLP.warm_up, repeats=1, label=f"ODE worker start-up + numba compile ({N_JOBS} processes)")
    log_timing(t_w, model="ODE", note="untimed in the samplers")
    rng = np.random.default_rng(SEED)
    th_prior = prior.sample(256, rng=rng)
    th_post = nuts_theta[rng.choice(len(nuts_theta), 256, replace=False)] if NUTS_REF else th_prior
    rows_thr = []
    for lab, th in (("prior draws", th_prior), ("posterior draws", th_post)):
        _, t = timeit_call(OLP.np_log_lik_theta, th, repeats=3, label=f"ODE log L, 256 {lab}")
        log_timing(t, n_evals=256, model="ODE", note="throughput test")
        rows_thr.append({"draws": lab, "solves_per_s": 256 / t.mean_s, "ms_per_solve_wall": 1e3 * t.mean_s / 256})
        LP_NN.precompile(max_rows=256, which=("ll",))
        _, t = timeit_call(LP_NN.np_log_lik_theta, th, repeats=5, label=f"{LABEL_NN} log L, 256 {lab}")
        log_timing(t, n_evals=256, model=LABEL_NN, note="throughput test")
        rows_thr[-1].update({"nn_evals_per_s": 256 / t.mean_s})
    thr = pd.DataFrame(rows_thr); thr["speed_up"] = thr.nn_evals_per_s / thr.solves_per_s
    thr.to_csv(OUT / "1_throughput.csv", index=False); display(thr.round(2))

    pts = {"best fit": theta_best} | ({"truth": truth_theta} if truth_theta is not None else {})
    tab = []
    fig, axs = plt.subplots(2, 1, figsize=(6, 5.5), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
    axs[0].semilogy(sur.qfls_used, 10 ** term_trpl.y_obs, "o", ms=3, color="k", label="data")
    for (lab, th), c in zip(pts.items(), ("C3", "c")):
        tt = OLP.trpl_terms(th[None]); y_nn = LP_NN.predict_trpl(th[None])["log10_tau_used"][0] if HAS_TRPL else None
        tab.append({"point": lab, "log L ODE": float(OLP.np_log_lik_theta(th[None])[0]),
                    "log L NN": float(LP_NN.np_log_lik_theta(th[None])[0]), "MSE ODE": float(tt["mse"][0]),
                    "MSE NN": float(np.mean((y_nn - term_trpl.y_obs) ** 2)), "rms NN-ODE": float(np.sqrt(np.nanmean((y_nn - tt["curves"][0]) ** 2)))})
        axs[0].semilogy(sur.qfls_used, 10 ** tt["curves"][0], "-", color=c, lw=1.5, label=f"ODE {lab}")
        axs[0].semilogy(sur.qfls_used, 10 ** y_nn, "--", color=c, lw=1.0, label=f"NN {lab}")
        axs[1].plot(sur.qfls_used, tt["curves"][0] - term_trpl.y_obs, "-", color=c, lw=1.2)
        axs[1].plot(sur.qfls_used, y_nn - term_trpl.y_obs, "--", color=c, lw=1.0)
    axs[1].axhline(0, color="0.3", lw=0.6); axs[1].set_xlabel(r"$\Delta E_\mathrm{F}$ (eV)"); axs[1].set_ylabel("model − data (dec.)")
    axs[0].set_ylabel(r"$\tau_\mathrm{diff}$ (s)"); axs[0].legend(fontsize=7)
    fig.tight_layout(); report.save_figure(fig, FIG / "1_ode_vs_nn_curves.png"); plt.show()
    tab = pd.DataFrame(tab); tab.to_csv(OUT / "1_loglik_nn_vs_ode.csv", index=False); display(tab.round(4))
else:
    print("mode 'sspl': no tr-PL term, no surrogate — the ODE part is skipped")

## 2. NN surrogate (mode `sspl`: exact model) with the same sampler settings

One XLA program for the SMC population (compiled untimed); emcee uses the padded numpy wrapper (precompiled).

In [ ]:
RES = {}
def export_weighted(res, name, model):
    th_w, w_w = res.flat_weighted()
    res.stages.to_csv(OUT / f"{name}_stages.csv", index=False)
    report.save_json({k: v for k, v in res.info.items() if k != "proposals"}, OUT / f"{name}_info.json")
    report.export_samples(res, prior, OUT / f"posterior_samples_{name}.csv")
    pd.concat([pd.DataFrame(th_w, columns=prior.names), pd.DataFrame({"weight": w_w, "island": res.weighted["group"]})],
              axis=1).to_csv(OUT / f"{name}_weighted_samples.csv", index=False, float_format="%.8g")

def corner_h5(name, theta, w, title):
    path = OUT / f"corner_{name}.h5"
    with h5py.File(path, "w") as f:
        f.attrs.update({"schema_version": 1, "method": name, "mode": MODE, "run_in": str(RUN_IN)})
        smc.corner_group(f.create_group("corner/physical"), theta, w, prior, theta_best, truth_theta, start_theta)
    with h5py.File(path, "r") as f:
        fig = cp.plot_corner(f["corner/physical"], title=title)
    report.save_figure(fig, FIG / f"corner_{name}_preview.png"); fig.savefig(FIG / f"corner_{name}_preview.pdf"); plt.show()
    return path

if RUN_NN:
    ev_nn = smc.PopulationEvaluator(LP_NN, max_batch=MAX_BATCH, fixed_shape=True)
    _, t_c = timeit_call(ev_nn.precompile, [MAX_BATCH], repeats=1, label="XLA compile of the NN population evaluator")
    log_timing(t_c, model=LABEL_NN, note="untimed in the samplers")
    RES["nn_smc"], t = timeit_call(smc.run_smc, LP_NN, **SMC, seed=SEED, evaluator=ev_nn, verbose=False,
                                   repeats=1, label=f"{LABEL_NN}: tempered SMC")
    si = RES["nn_smc"].info
    log_timing(t, n_evals=si["evals_total_points"], method="SMC", model=LABEL_NN,
               note=f"{si['n_islands']} islands x {si['n_particles']}, stages {si['n_stages']}")
    export_weighted(RES["nn_smc"], "nn_smc", LABEL_NN)
    print(f"{LABEL_NN} SMC: log Z = {RES['nn_smc'].log_evidence:.3f} ± {RES['nn_smc'].log_evidence_se:.3f}, Kish ESS {si['ess_weighted_kish']:.0f}")

    if RUN_NN_EMCEE:
        LP_NN.precompile(max_rows=EMCEE["n_walkers"], which=("lp",))
        init_w = samplers.init_dispersed(D, EMCEE["n_walkers"], np.random.default_rng(EMCEE["seed"]))
        RES["nn_emcee"], t = timeit_call(samplers.run_emcee, LP_NN, init_w, n_steps=EMCEE["n_steps"], burn_frac=EMCEE["burn_frac"],
                                         moves=EMCEE["moves"], seed=EMCEE["seed"], repeats=1, label=f"{LABEL_NN}: emcee")
        ei = RES["nn_emcee"].info
        log_timing(t, n_evals=ei["evals_total_points"], method="emcee", model=LABEL_NN,
                   note=f"{EMCEE['n_walkers']} walkers x {EMCEE['n_steps']} steps")
        report.export_samples(RES["nn_emcee"], prior, OUT / "posterior_samples_nn_emcee.csv")
        print(f"{LABEL_NN} emcee: acceptance {ei['acceptance_fraction']:.3f}, tau_int {np.round(ei['tau_int'], 1)}, steps/tau {ei['steps_over_tau']:.1f}")

## 3. ODE: tempered SMC

Every population (all islands) is ONE parallel batch of ODE solves.  The stage log is printed while it runs.

In [ ]:
if RUN_ODE and RUN_ODE_SMC:
    ev_ode = ol.OdePopulationEvaluator(OLP)
    OLP.time_ode_s, n0 = 0.0, OLP.n_uncovered
    RES["ode_smc"], t = timeit_call(smc.run_smc, OLP, **SMC, seed=SEED, evaluator=ev_ode, verbose=True,
                                    repeats=1, label="ODE: tempered SMC")
    si = RES["ode_smc"].info
    log_timing(t, n_evals=si["evals_total_points"], method="SMC", model="ODE",
               note=f"{si['n_islands']} islands x {si['n_particles']}, stages {si['n_stages']}; {N_JOBS} ODE workers; "
                    f"{OLP.n_uncovered - n0} solves not covering the data (log L = -inf); time in ODE calls {OLP.time_ode_s:.0f} s")
    export_weighted(RES["ode_smc"], "ode_smc", "ODE")
    th_w, w_w = RES["ode_smc"].flat_weighted()
    smc.write_h5(OUT / "ode_smc.h5", RES["ode_smc"], prior, best_theta=theta_best, truth_theta=truth_theta, start_theta=start_theta,
                 attrs={"run_in": str(RUN_IN), "mode": MODE, "model": "ODE (legacy solve_transient)"})
    corner_h5("ode_smc", th_w, w_w, f"{RUN_IN.name}: ODE, tempered SMC ({MODE})")
    print(f"ODE SMC: log Z = {RES['ode_smc'].log_evidence:.3f} ± {RES['ode_smc'].log_evidence_se:.3f}, Kish ESS {si['ess_weighted_kish']:.0f}, "
          f"{si['evals_total_points']:,} ODE solves in {t.mean_s / 60:.1f} min")

In [ ]:
if "ode_smc" in RES:
    st = RES["ode_smc"].stages
    fig, axs = plt.subplots(1, 3, figsize=(12, 3.2))
    for i, g in st.groupby("island"):
        axs[0].semilogy(g.stage, g.beta, "o-", ms=3, label=f"island {i}")
        axs[1].plot(g.stage, g.acceptance, "o-", ms=3)
        axs[2].plot(g.stage, g.time_stage_s, "o-", ms=3)
    if "nn_smc" in RES:
        for i, g in RES["nn_smc"].stages.groupby("island"):
            axs[0].semilogy(g.stage, g.beta, "x:", ms=3, color="0.5", label="NN" if i == 0 else None)
    axs[0].set_ylabel("β"); axs[1].set_ylabel("MH acceptance"); axs[2].set_ylabel("wall time per stage (s)")
    for a in axs: a.set_xlabel("stage"); a.grid(alpha=0.3)
    axs[0].legend(fontsize=7); fig.suptitle(f"ODE tempered SMC ({MODE})", fontsize=9)
    fig.tight_layout(); report.save_figure(fig, FIG / "3_ode_smc_stages.png"); plt.show()

## 4. ODE: emcee (same walkers, steps and seed as the NN run)

In [ ]:
if RUN_ODE and RUN_ODE_EMCEE:
    init_w = samplers.init_dispersed(D, EMCEE["n_walkers"], np.random.default_rng(EMCEE["seed"]))
    OLP.time_ode_s, n0 = 0.0, OLP.n_uncovered
    RES["ode_emcee"], t = timeit_call(samplers.run_emcee, OLP, init_w, n_steps=EMCEE["n_steps"], burn_frac=EMCEE["burn_frac"],
                                      moves=EMCEE["moves"], seed=EMCEE["seed"], repeats=1, label="ODE: emcee")
    ei = RES["ode_emcee"].info
    log_timing(t, n_evals=ei["evals_total_points"], method="emcee", model="ODE",
               note=f"{EMCEE['n_walkers']} walkers x {EMCEE['n_steps']} steps; {N_JOBS} ODE workers; "
                    f"{OLP.n_uncovered - n0} solves not covering the data; time in ODE calls {OLP.time_ode_s:.0f} s")
    report.export_samples(RES["ode_emcee"], prior, OUT / "posterior_samples_ode_emcee.csv")
    report.save_json(ei, OUT / "ode_emcee_info.json")
    th = RES["ode_emcee"].flat_theta()
    corner_h5("ode_emcee", th, np.ones(len(th)), f"{RUN_IN.name}: ODE, emcee ({MODE})")
    print(f"ODE emcee: acceptance {ei['acceptance_fraction']:.3f}, tau_int {np.round(ei['tau_int'], 1)}, steps/tau {ei['steps_over_tau']:.1f}, "
          f"{ei['evals_total_points']:,} ODE solves in {t.mean_s / 60:.1f} min")

## 5. Comparison: marginals, two-sample tests, overlay corner, HDF5 corners of every method

Reference = NN NUTS of `RUN_IN` (if `RUN_IN` has this mode), else NN SMC.  Deviations in units of the reference
posterior sd.  C2ST: held-out accuracy of a classifier separating two sample sets (0.5 = indistinguishable).

In [ ]:
sets = {}
if NUTS_REF:
    sets[f"{LABEL_NN} NUTS (main run)"] = (nuts_theta, None)
for k, lab in (("nn_smc", f"{LABEL_NN} SMC"), ("ode_smc", "ODE SMC")):
    if k in RES:
        sets[lab] = RES[k].flat_weighted()
for k, lab in (("nn_emcee", f"{LABEL_NN} emcee"), ("ode_emcee", "ODE emcee")):
    if k in RES:
        sets[lab] = (RES[k].flat_theta(), None)
marg = smc.marginal_table(sets, prior); marg.to_csv(OUT / "5_marginals.csv", index=False)
ref_lab = next(iter(sets))
ref = marg[marg.method == ref_lab].set_index("parameter")
dev = []
for m_, g in marg.groupby("method"):
    g = g.set_index("parameter")
    dev.append({"method": m_, **{f"max |Δ{q}| / sd_ref": float(((g[q] - ref.loc[g.index, q]).abs() / ref.loc[g.index, "sd"]).max())
                                 for q in ("q2.5", "q16", "q50", "q84", "q97.5")},
                "max |1 - sd/sd_ref|": float((1 - g["sd"] / ref.loc[g.index, "sd"]).abs().max())})
dev = pd.DataFrame(dev).set_index("method"); dev.to_csv(OUT / "5_deviation_from_reference.csv")
print(f"reference: {ref_lab}"); display(marg.round(4)); display(dev.round(3))

Xref = prior.to_display(sets[ref_lab][0])[:, free]
c2 = {}
for lab, (th, w) in sets.items():
    if lab != ref_lab:
        c2[f"{lab} vs {ref_lab}"] = smc.c2st(Xref, prior.to_display(th)[:, free], w_b=w, seed=0)
if "nn_smc" in RES and "ode_smc" in RES:
    thn, wn = RES["nn_smc"].flat_weighted(); tho, wo = RES["ode_smc"].flat_weighted()
    c2["ODE SMC vs NN SMC (final particles)"] = smc.c2st(prior.to_display(RES["nn_smc"].flat_theta())[:, free],
                                                         prior.to_display(RES["ode_smc"].flat_theta())[:, free], seed=0)
c2 = pd.Series(c2, name="C2ST accuracy"); c2.to_csv(OUT / "5_c2st.csv"); display(c2.round(3))

In [ ]:
# overlay corner (display coordinates): reference filled, ODE SMC (or ODE emcee) contours
ode_key = "ode_smc" if "ode_smc" in RES else ("ode_emcee" if "ode_emcee" in RES else None)
over_key = ode_key or ("nn_smc" if "nn_smc" in RES else None)
if over_key is not None:
    r_ = RES[over_key]
    over = smc.resample_weighted(*r_.flat_weighted(), 20000, seed=1) if hasattr(r_, "weighted") and r_.weighted else r_.flat_theta()
    fig = report.plot_corner(prior.to_display(sets[ref_lab][0]), prior,
                             truth=None if truth_theta is None else prior.to_display(truth_theta[None])[0],
                             best=prior.to_display(theta_best[None])[0], overlay=prior.to_display(over),
                             overlay_label={"ode_smc": "ODE SMC", "ode_emcee": "ODE emcee", "nn_smc": f"{LABEL_NN} SMC"}[over_key],
                             label=ref_lab)
    report.save_figure(fig, FIG / f"5_corner_overlay_{MODE}.png"); fig.savefig(FIG / f"5_corner_overlay_{MODE}.pdf"); plt.show()

# HDF5 corners of the NN methods (the ODE ones were written in sections 3 and 4)
H5 = {k: OUT / f"corner_{k}.h5" for k in ("ode_smc", "ode_emcee") if (OUT / f"corner_{k}.h5").exists()}
if NUTS_REF:
    H5["nn_nuts"] = corner_h5("nn_nuts", nuts_theta, np.ones(len(nuts_theta)), f"{RUN_IN.name}: {LABEL_NN}, NUTS ({MODE})")
if "nn_smc" in RES:
    H5["nn_smc"] = corner_h5("nn_smc", *RES["nn_smc"].flat_weighted(), f"{RUN_IN.name}: {LABEL_NN}, tempered SMC ({MODE})")
if "nn_emcee" in RES:
    th = RES["nn_emcee"].flat_theta()
    H5["nn_emcee"] = corner_h5("nn_emcee", th, np.ones(len(th)), f"{RUN_IN.name}: {LABEL_NN}, emcee ({MODE})")
print({k: str(v) for k, v in H5.items()})

## 6. Timing table and surrogate speed-up

* identical settings: wall time ODE / NN for the same sampler, seed and settings (SMC, emcee);
* per evaluation: (time / evaluations) ODE vs NN (the samplers need different numbers of evaluations);
* full `main.ipynb` samplers: ODE time ESTIMATED from their NN evaluation counts and the measured ODE time per
  evaluation of the ODE SMC run in this notebook (large parallel batches: the best case) (NUTS: a lower bound — it would also need gradients, which the ODE does not provide).

In [ ]:
timing = pd.DataFrame(TIMINGS)
timing["time"] = [format_mean_sd(m, s, int(n), "s") for m, s, n in zip(timing.mean_s, timing.std_s, timing.repeats)]
timing["ms_per_eval"] = 1e3 * timing.mean_s / timing.n_evals
timing.to_csv(OUT / "6_timing.csv", index=False)
display(timing[["label", "model", "method", "time", "n_evals", "ms_per_eval", "note"]])

rows = []
for meth in ("SMC", "emcee"):
    a = timing[(timing.method == meth) & (timing.model == LABEL_NN)]
    b = timing[(timing.method == meth) & (timing.model == "ODE")]
    if len(a) and len(b):
        a, b = a.iloc[0], b.iloc[0]
        rows.append({"sampler": meth, "time_nn_s": a.mean_s, "time_ode_s": b.mean_s, "evals_nn": a.n_evals, "evals_ode": b.n_evals,
                     "speed_up_wall": b.mean_s / a.mean_s, "ms_per_eval_nn": a.ms_per_eval, "ms_per_eval_ode": b.ms_per_eval,
                     "speed_up_per_eval": b.ms_per_eval / a.ms_per_eval})
speed = pd.DataFrame(rows)
# ODE time per evaluation for the estimates: the SMC run (large parallel batches = best case), else emcee
_o = timing[(timing.model == "ODE") & timing.method.notna()].set_index("method")
ms_ode = float(_o.loc["SMC", "ms_per_eval"] if "SMC" in _o.index else (_o.loc["emcee", "ms_per_eval"] if "emcee" in _o.index else np.nan))
main_tm = pd.read_csv(RUN_IN / "11_timing.csv") if (RUN_IN / "11_timing.csv").exists() else None
est = []
if main_tm is not None and np.isfinite(ms_ode):
    for lab, pat in (("NUTS (main run)", "^NUTS \\(warm-up"), ("emcee (main run)", "^emcee"), ("grid (main run)", "^grid")):
        r_ = main_tm[main_tm.label.str.contains(pat, regex=True)]
        if len(r_) and np.isfinite(r_.iloc[0].n_evals):
            r_ = r_.iloc[0]
            est.append({"sampler": lab, "time_nn_s": r_.mean_s, "evals_nn": r_.n_evals,
                        "time_ode_est_s": r_.n_evals * ms_ode / 1e3, "speed_up_est": r_.n_evals * ms_ode / 1e3 / r_.mean_s,
                        "note": "lower bound: NUTS also needs gradients" if lab.startswith("NUTS") else ""})
est = pd.DataFrame(est)
speed.to_csv(OUT / "6_speedup_same_settings.csv", index=False); est.to_csv(OUT / "6_ode_time_estimates_main_samplers.csv", index=False)
display(speed.round(3)); display(est.round(1))

summary = {"run_in": str(RUN_IN), "mode": MODE, "sigmas": SIG, "n_jobs": N_JOBS, "cpu_count": os.cpu_count(),
           "ms_per_ode_eval_in_samplers": ms_ode, "speed_up_same_settings": speed.to_dict("records"),
           "ode_estimates_main_samplers": est.to_dict("records"), "deviation_from_reference": dev.to_dict("index"),
           "reference": ref_lab, "c2st": c2.to_dict(),
           "log_z": {k: [RES[k].log_evidence, RES[k].log_evidence_se] for k in ("nn_smc", "ode_smc") if k in RES},
           "diagnostics": {k: {"max_r_hat": float(samplers.diagnostics(RES[k], prior).r_hat.max()),
                               "min_ess_bulk": float(samplers.diagnostics(RES[k], prior).ess_bulk.min())} for k in RES},
           "h5": {k: str(v) for k, v in H5.items()}}
report.save_json(summary, OUT / "summary.json")
print("written:", OUT)